# NOTEBOOK_Data-eau_00_API

### **Descriptif du Notebook VERSION VS CODE** : 

**Le projet :**
- Connexion à l'API Hub Eau sur le jeu de donnée "Piézométrie" (https://hubeau.eaufrance.fr/page/api-piezometrie)
- Données recherchées :  
    A. La liste des stations autour de Valence 50km `/stations`  
    B. Les station et leur niveau d'eau. `/chroniques`
- Zone géographique : le bassion de Valence (26000) = utiliser une bbox « carré de 100 km centré sur Valence »

**Les étapes du Notebook :**

1. Imports 
2. Configuration : avec variables du projet (.env) + variables métier :
    - fichier .env = URL de l'API , Token (Identifiant pour l'API, ici pas nécessaire)
    - constantes/variables métier = Latitude/Longtude de Valence, demi_coté_carré de la Bbox
3. Calcul de la bbox
4. Fonction d'appel à l'API : gestion des erreurs + pagination
5. Récupération des stations piézométriques
6. Mise en DataFrame et premier contrôle (shape, colonnes, valeurs manquantes)
7. Sauvegarde des données brutes (data/raw/)
    

### Prérequis avant d'utiliser le notebook

**Première installation (une seule fois)**
- Cloner le dépôt, puis dans le terminal, à la racine du projet :
  `python3 -m venv .venv` → `source .venv/bin/activate` → `pip install -r requirements.txt`
- Créer le fichier de variables : `cp .env.example .env`

**À chaque session de travail**
- Ouvrir le **dossier racine** `Hub-eau` dans VS Code (pas seulement le notebook)
- Récupérer les dernières modifications : `git pull`
- Vérifier que le **kernel** sélectionné (en haut à droite) est `.venv`
- Exécuter la cellule de vérification ci-dessous

**Pour installer une nouvelle bibliothèque**
- Dans le terminal, venv activé (`(.venv)` visible) : `source .venv/bin/activate`
- `pip install <bibliothèque>`, puis `pip freeze > requirements.txt`, puis commit

**Avant chaque commit**
- *Restart Kernel* + *Run All* : le notebook doit tourner en entier sans erreur
- `git status` : `.env` et `.venv/` ne doivent jamais apparaître


In [5]:
# Vérification de l'environnement : le kernel doit être celui du .venv du projet
import sys

print("Python utilisé :", sys.executable)
assert ".venv" in sys.executable, "⚠️ Mauvais kernel : sélectionner le .venv en haut à droite"

Python utilisé : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/.venv/bin/python


### **Etapes du Notebook:**

In [7]:
# 1. Imports des bibliothèques

# Bibliothèque standard (livrée avec Python)
import math                         # calcul de la bbox (cosinus de la latitude)
import os                           # lecture des variables d'environnement
import time                         # pause entre deux appels à l'API
from datetime import datetime       # horodatage des fichiers sauvegardés
from pathlib import Path            # chemins de fichiers (data/raw/...)

# Bibliothèques externes (installées dans le .venv, listées dans requirements.txt)
import pandas as pd                 # mise en DataFrame et contrôle des données
import requests                     # appels HTTP à l'API Hub'Eau
from dotenv import load_dotenv      # chargement du fichier .env


In [8]:
# 2. Configuration

# --- A. Variables d'environnement (fichier .env) ---
load_dotenv()                                   # charge le .env (cherché en remontant les dossiers)

BASE_URL = os.getenv("HUBEAU_BASE_URL")
if BASE_URL is None:
    raise ValueError("HUBEAU_BASE_URL introuvable : vérifier que le fichier .env existe à la racine du projet")

URL_STATIONS = f"{BASE_URL}/stations"           # liste des stations piézométriques
URL_CHRONIQUES = f"{BASE_URL}/chroniques"       # mesures de niveau d'eau par station

# --- B. Constantes métier ---
VALENCE_LAT = 44.9334                           # latitude du centre de la zone (WGS84)
VALENCE_LON = 4.8924                            # longitude du centre de la zone (WGS84)
DEMI_COTE_KM = 50                               # demi-côté du carré => carré de 100 km

# --- C. Paramètres techniques de l'API ---
PAGE_SIZE = 1000                                # nombre de résultats par page
TIMEOUT_S = 30                                  # délai max d'attente d'une réponse (secondes)
PAUSE_S = 0.2                                   # pause entre deux pages (respect de l'API)

# --- D. Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)     # créé s'il n'existe pas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Récapitulatif ---
print("URL stations   :", URL_STATIONS)
print("Centre         :", VALENCE_LAT, VALENCE_LON, f"(demi-côté {DEMI_COTE_KM} km)")
print("Racine projet  :", PROJECT_ROOT)
print("Données brutes :", DATA_RAW)


URL stations   : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations
Centre         : 44.9334 4.8924 (demi-côté 50 km)
Racine projet  : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau
Données brutes : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/data/raw


In [9]:
# 3. Calcul de la bbox

# Définir la zone concernée par les données = BBox autour de la ville choisie (Ici = Valence /50 km)
    # **Définir la zone BBox** (zone carré de 50 km autour de Valence avec code communes) : Rectangle d'emprise de l'objet demandé, emprise au format : min longitude, min latitude, max longitude, max latitude avec les coordonnées en WGS84 (EPSG:4326), le point doit être utilisé comme séparateur décimal, exemple : 1.6194,47.7965,2.1910,47.9988.  
    # Vérifier sur : https://bboxfinder.com/#44.484200,4.258000,45.382600,5.526800

def bbox_autour(lat, lon, demi_cote_km):
    """Renvoie la BBox au format Hub'Eau : min_lon,min_lat,max_lon,max_lat"""
    d_lat = demi_cote_km / 111.32
    d_lon = demi_cote_km / (111.32 * math.cos(math.radians(lat)))
    return f"{lon - d_lon:.4f},{lat - d_lat:.4f},{lon + d_lon:.4f},{lat + d_lat:.4f}"

BBOX = bbox_autour(VALENCE_LAT, VALENCE_LON, DEMI_COTE_KM)
print("BBox :", BBOX)

BBox : 4.2579,44.4842,5.5269,45.3826


In [ ]:
# 4. Fonction d'appel à l'API : gestion des erreurs + pagination


#- Récupérer les paramètres de l'API : 
#    - Pagination : page, limit_page, limite_rate ?
#    - Filtre : quel champs précis ?
#    - Format : json, autre ?
#    => définir ces paramètres dans le code. 

In [ ]:
# 5. Récupération des stations piézométriques

In [ ]:
# 6. Mise en DataFrame et premier contrôle (shape, colonnes, valeurs manquantes)

In [ ]:
# 7. Sauvegarde des données brutes (data/raw/)